# Dynamic prompts and templates for better control

You are building an AI Content processor that must handle various task types - summarization, rewriting, explaining through a single Langchain pipeline.

Build parameterized prompt templates, add few shot examples based on the input. Route to different system styles or tasks with branching logic.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("AZURE_OPENAI_KEY")
api_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_VERSION")


In [3]:
from typing import List, Optional
from pydantic import BaseModel, Field
from datetime import datetime
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
import pandas as pd

In [4]:
default_temp = 0.2
prompt_version = "v1.3"

In [5]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint= api_endpoint,
    azure_deployment="gpt-4o-mini",
    openai_api_key=api_key,
    openai_api_version=api_version,
    temperature=default_temp)

In [10]:
from langchain_core.output_parsers import PydanticOutputParser


class ControlledResponse(BaseModel):
    task: str = Field( description="task type: summarize, rewrite, explain")
    version: str = Field( description="version of the prompt template to use")
    style: str = Field(description="style of the response, e.g. concise, detailed, formal, informal")
    bullets: Optional[List[str]] = Field(default=None, description="short bullet points if task-summarize or explain")
    text: Optional[str] = Field(default=None, description="rewritten text if task-rewrite or final text")

parser = PydanticOutputParser(pydantic_object=ControlledResponse)



In [7]:
def temperature_for(risk_level:str) ->float:
    table = {"low": 0.2, "medium": 0.5, "high": 0.8}
    return table.get(risk_level.lower(),default_temp)
def style_token(user_style:Optional[str]) -> str:
    style = user_style if user_style else "concise, professional"
    return f"style: {style}"

## Few shot examples

In [8]:
examples = [
    {"task":"summarize","input":"unit testing benefits","output":"- Catches regressions early\n - Documents code behavior" },
    {"task":"rewrite","input":"Close door, loud.","output":"Could you please close the door? It's quite loud."},
    {"task":"explain","input":"concurrency vs parallelism","output":"- Concurrency is about dealing with multiple tasks at once, while parallelism is about doing multiple tasks at the same time.\n - Concurrency can be achieved with a single core by interleaving tasks, while parallelism requires multiple cores."}
]
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
example_prompt = PromptTemplate.from_template("Task: {task}\nInput: {input}\nOutput: {output}\n")
selector = LengthBasedExampleSelector(examples=examples,example_prompt=example_prompt, max_input_length=200) 
fewshot = FewShotPromptTemplate(example_selector=selector, example_prompt=example_prompt, prefix="Here are some examples:",
                                 suffix="Task:{task}\nInput:{user_input}. Follow the instructions from above.",
                                 input_variables=["task","user_input"])

# Takes all examples
# Formats them using example_prompt
# Adds them one by one
# Stops when total length exceeds max_length

In [9]:
from langchain_core.output_parsers import StrOutputParser
chain = fewshot | llm | StrOutputParser()
response = chain.invoke({
    "task": "summarize",
    "user_input": "benefits of logging in applications"
})

print(response)

- Helps in debugging by providing detailed error information  
- Aids in monitoring application performance and behavior  
- Facilitates auditing and tracking user activities  
- Supports identifying security issues and breaches


In [12]:
summarize_helper = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that summarizes text. Extract key points and present them in 2-4 bullet form."),
    ]
)
rewrite_helper = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that rewrites text. Rewrite the input text to be more clear with the same meaning."),
    ])
explain_helper = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that explains concepts. Provide a clear and concise explanation with key points in 2-4 bullet form."),
    ])

## JSON Format Instructions

In [20]:
def json_instructions():
    return """Respond in a JSON format with the following keys:
    - task: one of summarize, rewrite, explain
    - version: prompt template version, e.g. v1.0
    - style: style of the response, e.g. concise, detailed, formal, informal
    - bullets: (optional) short bullet points if task is summarize or explain
    - text: (optional) rewritten text if task is rewrite or final text"""

## Build a dynamic prompt chain

In [22]:
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnablePassthrough
from langchain_core.messages import HumanMessage, SystemMessage
#helper functions
#few shot examples
#format instructions
#output parser

def build_chain(helper_prompt):
    return (
       RunnableLambda(lambda x: {
            "fewshot":fewshot.format(task=x["task"], user_input=x["user_input"]),
            "input":x["user_input"],
            "format_instructions":json_instructions(),
        }) |
        ChatPromptTemplate.from_messages(
            helper_prompt.messages + [("human", "{fewshot} \n\nInput:{input}\n\n{format_instructions}")]) | llm | parser
    )

summarize_chain = build_chain(summarize_helper)
rewrite_chain = build_chain(rewrite_helper)
explain_chain = build_chain(explain_helper)

router = RunnableBranch(
    (lambda x: x["task"].lower() == "summarize", summarize_chain),
    (lambda x: x["task"].lower() == "rewrite", rewrite_chain),
    (lambda x: x["task"].lower() == "explain", explain_chain),
    summarize_chain
)

# =========================
# RUN
# =========================
response = router.invoke({
    "task": "explain",
    "user_input": "difference between multiprocessing and multithreading"
})

print(response)

task='explain' version='v1.0' style='concise' bullets=['Multiprocessing uses multiple processes with separate memory spaces, while multithreading uses multiple threads within the same process sharing memory.', 'Multiprocessing avoids issues like the Global Interpreter Lock (GIL) in Python, enabling true parallelism on multiple CPU cores.', 'Multithreading is lighter weight and better for I/O-bound tasks but can face synchronization challenges and race conditions.', 'Multiprocessing has higher overhead due to process creation and inter-process communication.'] text=None
